# Lab 10B ? Delta Change Data Feed (CDF)

The source and target below are the only tables this notebook writes. Credentials are not included.

> The verified Academy run is already complete. The replay cells below use `CREATE OR REPLACE` to reset these two Lab 10 tables before replaying the demonstration. Do not rerun them if you need to preserve the current Delta history.

## Create a CDF-enabled source and seed the initial five customers

In [ ]:
%sql
CREATE OR REPLACE TABLE dbr_dev.ayyuborujzade_bronze.lab10_customers_cdc (
  customer_id INT,
  customer_name STRING,
  country STRING,
  segment STRING
) USING DELTA
TBLPROPERTIES (delta.enableChangeDataFeed = true)

In [ ]:
%sql
SHOW TBLPROPERTIES dbr_dev.ayyuborujzade_bronze.lab10_customers_cdc ('delta.enableChangeDataFeed')

Verified property: `delta.enableChangeDataFeed = true`.

In [ ]:
%sql
INSERT INTO dbr_dev.ayyuborujzade_bronze.lab10_customers_cdc
SELECT customer_id, customer_name, country, segment
FROM lab10_postgres_catalog.public.customers

In [ ]:
%sql
SELECT customer_id, customer_name, country, segment
FROM dbr_dev.ayyuborujzade_bronze.lab10_customers_cdc
ORDER BY customer_id

The seed loaded **5 initial customers** from the foreign PostgreSQL table.

## Initialize the Silver target from the initial source state

In [ ]:
%sql
CREATE OR REPLACE TABLE dbr_dev.ayyuborujzade_silver.lab10_customers_target
AS SELECT customer_id, customer_name, country, segment
FROM dbr_dev.ayyuborujzade_bronze.lab10_customers_cdc

## Simulate INSERT, UPDATE, and DELETE changes

In [ ]:
%sql
INSERT INTO dbr_dev.ayyuborujzade_bronze.lab10_customers_cdc
VALUES (6, 'John Miller', 'Spain', 'Consumer')

In [ ]:
%sql
UPDATE dbr_dev.ayyuborujzade_bronze.lab10_customers_cdc
SET segment = 'Consumer'
WHERE customer_id = 2

In [ ]:
%sql
DELETE FROM dbr_dev.ayyuborujzade_bronze.lab10_customers_cdc
WHERE customer_id = 5

## Read the Change Data Feed

In [ ]:
%sql
SELECT customer_id, customer_name, country, segment,
       _change_type, _commit_version, _commit_timestamp
FROM table_changes(
  'dbr_dev.ayyuborujzade_bronze.lab10_customers_cdc', 0
)
ORDER BY _commit_version, customer_id, _change_type

Observed CDF event counts:

| `_change_type` | Events |
|---|---:|
| `insert` | 6 (five seed rows plus customer 6) |
| `update_preimage` | 1 |
| `update_postimage` | 1 |
| `delete` | 1 |

## Incrementally merge the latest events into Silver

In [ ]:
%sql
MERGE INTO dbr_dev.ayyuborujzade_silver.lab10_customers_target AS target
USING (
  SELECT customer_id, customer_name, country, segment, _change_type
  FROM table_changes(
    'dbr_dev.ayyuborujzade_bronze.lab10_customers_cdc', 0
  )
  WHERE _change_type IN ('insert', 'update_postimage', 'delete')
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY customer_id ORDER BY _commit_version DESC
  ) = 1
) AS changes
ON target.customer_id = changes.customer_id
WHEN MATCHED AND changes._change_type = 'delete' THEN DELETE
WHEN MATCHED THEN UPDATE SET
  target.customer_name = changes.customer_name,
  target.country = changes.country,
  target.segment = changes.segment
WHEN NOT MATCHED AND changes._change_type <> 'delete' THEN INSERT (
  customer_id, customer_name, country, segment
) VALUES (
  changes.customer_id, changes.customer_name, changes.country, changes.segment
)

Verified MERGE result: **6 affected rows** ? 4 updated, 1 deleted, and 1 inserted.

## Verify the final target

In [ ]:
%sql
SELECT customer_id, customer_name, country, segment
FROM dbr_dev.ayyuborujzade_silver.lab10_customers_target
ORDER BY customer_id

Verified final target (**5 rows**):

| ID | Customer | Country | Segment |
|---:|---|---|---|
| 1 | Alice Johnson | Poland | Consumer |
| 2 | Mark Smith | Germany | Consumer |
| 3 | Sofia Brown | Poland | Consumer |
| 4 | Daniel Wilson | France | Corporate |
| 6 | John Miller | Spain | Consumer |

In [ ]:
%sql
SELECT COUNT(*) AS customer_5_rows
FROM dbr_dev.ayyuborujzade_silver.lab10_customers_target
WHERE customer_id = 5

Verified `customer_5_rows = 0`.

In [ ]:
%sql
SELECT COUNT(*) AS customer_6_rows
FROM dbr_dev.ayyuborujzade_silver.lab10_customers_target
WHERE customer_id = 6

Verified `customer_6_rows = 1`.

In [ ]:
%sql
SELECT COUNT(*) AS mismatched_rows
FROM (
  SELECT customer_id, customer_name, country, segment
  FROM dbr_dev.ayyuborujzade_bronze.lab10_customers_cdc
  EXCEPT
  SELECT customer_id, customer_name, country, segment
  FROM dbr_dev.ayyuborujzade_silver.lab10_customers_target
  UNION ALL
  SELECT customer_id, customer_name, country, segment
  FROM dbr_dev.ayyuborujzade_silver.lab10_customers_target
  EXCEPT
  SELECT customer_id, customer_name, country, segment
  FROM dbr_dev.ayyuborujzade_bronze.lab10_customers_cdc
) AS differences

Verified `mismatched_rows = 0`; source and target matched in both directions.

## Batch ingestion, CDC, SCD, and late-arriving data

Batch ingestion copies a snapshot or range on a schedule. CDC captures row-level changes so downstream tables can process inserts, updates, and deletes incrementally. Delta CDF is one way to read committed changes from a Delta table.

CDC describes how changes are captured and delivered; SCD describes how a target models them. This merge maintains a current-state (SCD Type 1-style) target. An SCD Type 2 design would preserve historical versions instead. For recurring production processing, persist the last applied commit version and define how late or out-of-order business events are deduplicated and reconciled.